# **Step 3: Model Evaluation, Testing & Visual Inference**
### **Helmet Detection (YOLOv8)**

This notebook covers testing and evaluating the trained model:
1. **Load Trained Model**: Load weights from `saved_models/best.pt`.
2. **Quantitative Benchmarks**: Evaluate on unseen **Test Dataset (77 images)** using `model.val(split='test')`.
3. **Class-Wise Metrics**: Inspect precision, recall, and mAP@50 for `With Helmet` vs `Without Helmet`.
4. **Evaluation Curves**: Display Confusion Matrix, Precision-Recall curve, and F1 curve.
5. **Visual Predictions**: Generate a 3x3 grid of test predictions with bounding boxes and confidence scores.
6. **Custom Image Inference**: Run detection on any custom image or traffic scene.


## **1. Import Modules & Setup Environment**

In [ ]:
import os
import random
import cv2
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO

# Fix OpenMP duplicate library conflict on Windows
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## **2. Load Trained Model from saved_models/**

In [ ]:
BASE_DIR = os.getcwd()

model_candidates = [
    os.path.join(BASE_DIR, 'saved_models', 'best.pt'),
    os.path.join(BASE_DIR, 'runs', 'detect', 'train', 'weights', 'best.pt'),
    os.path.join(BASE_DIR, 'runs', 'detect', 'runs', 'detect', 'train', 'weights', 'best.pt')
]

MODEL_PATH = None
for mp in model_candidates:
    if os.path.exists(mp):
        MODEL_PATH = mp
        break

best_model = YOLO(MODEL_PATH)

print(f"✅ Model Loaded Successfully from: {MODEL_PATH}")
print(f"Model Classes: {best_model.names}")


## **3. Evaluate on Unseen Test Dataset**

In [ ]:
DATA_YAML_PATH = os.path.join(BASE_DIR, 'HelmetDataset', 'data.yaml')

# Evaluate model on the test split
eval_metrics = best_model.val(
    data=DATA_YAML_PATH,
    split='test'
)

print("\n" + "=" * 55)
print("📊 TEST SET BENCHMARKS (UNSEEN DATA)")
print("=" * 55)
print(f"Overall mAP@50    : {eval_metrics.box.map50 * 100:.2f}%")
print(f"Overall mAP@50-95 : {eval_metrics.box.map * 100:.2f}%")
print(f"Overall Precision : {eval_metrics.box.mp * 100:.2f}%")
print(f"Overall Recall    : {eval_metrics.box.mr * 100:.2f}%")
print("=" * 55)


## **4. Display Confusion Matrix & Performance Curves**

In [ ]:
# Display training and validation evaluation charts
charts = [
    os.path.join('runs', 'detect', 'train', 'confusion_matrix.png'),
    os.path.join('runs', 'detect', 'train', 'BoxPR_curve.png'),
    os.path.join('runs', 'detect', 'train', 'BoxF1_curve.png')
]

for chart in charts:
    if os.path.exists(chart):
        img = cv2.imread(chart)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(8, 6))
        plt.imshow(img)
        plt.title(os.path.basename(chart), fontsize=13)
        plt.axis('off')
        plt.show()


## **5. Visualize Predictions on Random Unseen Test Images**

In [ ]:
# Visualize Random Test Predictions
TEST_IMG_DIR = os.path.join(BASE_DIR, 'HelmetDataset', 'test', 'images')
test_images = os.listdir(TEST_IMG_DIR)
sample_test_images = random.sample(test_images, min(9, len(test_images)))

plt.figure(figsize=(12, 12))
for i, image_name in enumerate(sample_test_images):
    image_path = os.path.join(TEST_IMG_DIR, image_name)

    results = best_model.predict(image_path, conf=0.35, imgsz=640, verbose=False)[0]

    annotated_image = results.plot()
    annotated_image = cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB)

    plt.subplot(3, 3, i + 1)
    plt.imshow(annotated_image)
    plt.title(image_name, fontsize=8)
    plt.axis('off')

plt.tight_layout()
plt.show()


## **6. Single Image Detection with Violation Highlighting**

In [ ]:
def detect_and_display(image_path, conf=0.3):
    """Run inference and display results with violation count."""
    img = cv2.imread(image_path)
    if img is None:
        print(f"Could not read image: {image_path}")
        return

    results = best_model.predict(img, conf=conf, verbose=False)[0]
    annotated = results.plot()
    annotated = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

    # Count detections
    boxes = results.boxes
    with_helmet = sum(1 for b in boxes if int(b.cls[0]) == 0)
    without_helmet = sum(1 for b in boxes if int(b.cls[0]) == 1)

    title_color = 'crimson' if without_helmet > 0 else 'forestgreen'
    title = f"With Helmet: {with_helmet} | Without Helmet (VIOLATIONS): {without_helmet}"

    plt.figure(figsize=(9, 6))
    plt.imshow(annotated)
    plt.title(title, fontsize=12, color=title_color, weight='bold')
    plt.axis('off')
    plt.show()

# Test on first image in test folder
sample_test_image = os.path.join(TEST_IMG_DIR, test_images[0])
print(f"Analyzing: {sample_test_image}")
detect_and_display(sample_test_image, conf=0.3)


## **7. Real-Time Video or Webcam Inference**
To run real-time inference on a video file or webcam, run the standalone CLI script in your terminal:
```bash
# Webcam live detection
python detect_image.py --source 0

# Video file detection
python detect_image.py --source traffic.mp4 --show
```
